In [1]:
!git clone https://github.com/lshek22/facial-expression-recognition.git
%cd facial-expression-recognition
!pip install wandb -q

fatal: destination path 'facial-expression-recognition' already exists and is not an empty directory.
/content/facial-expression-recognition


In [2]:
import os, json
from google.colab import userdata

os.makedirs('/root/.config/kaggle', exist_ok=True)
kaggle_creds = {
    "username": 'lukashekiladze',
    "key": 'b2f5f196e6fac2129c67a38910282180'
}
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.system('chmod 600 /root/.config/kaggle/kaggle.json')

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip
!ls

100% 285M/285M [00:14<00:00, 20.6MB/s]

challenges-in-representation-learning-facial-expression-recognition-challenge.zip
example_submission.csv
experiment_deep_cnn.ipynb
experiment_simple_cnn.ipynb
fer2013.tar.gz
icml_face_data.csv
README.md
test.csv
train.csv


In [3]:
import wandb
wandb.login(key='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')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lshek22 (lshek22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

train_df = pd.read_csv('train.csv')
train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42)

class FERDataset(Dataset):
    def __init__(self, df, transform=None):
        self.labels = df['emotion'].values
        self.images = df['pixels'].apply(
            lambda x: np.array(x.split(), dtype=np.float32).reshape(48, 48)
        ).values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx] / 255.0
        img = torch.tensor(img).unsqueeze(0)
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
])

train_dataset = FERDataset(train_data, transform=train_transform)
val_dataset   = FERDataset(val_data)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Device: {device}")

Train: 25838, Val: 2871, Device: cuda


In [5]:
import torch.nn as nn
import torchvision.models as models

class ResNet18FER(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(ResNet18FER, self).__init__()
        self.resnet = models.resnet18(pretrained=True)


        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, 7)
        )

    def forward(self, x):
        return self.resnet(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = ResNet18FER(dropout_rate=0.5).to(device)
print(model)

Using: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


ResNet18FER(
  (resnet): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_

In [6]:
import math

model_check = ResNet18FER().to(device)
sample_imgs, sample_labels = next(iter(train_loader))
sample_imgs = sample_imgs.to(device)

with torch.no_grad():
    out = model_check(sample_imgs)

loss_val = nn.CrossEntropyLoss()(out, sample_labels.to(device))
print(f"Output shape : {out.shape}")
print(f"Initial loss : {loss_val.item():.4f} (expected ~{math.log(7):.4f})")

model_check.train()
opt = torch.optim.Adam(model_check.parameters(), lr=0.001)
opt.zero_grad()
out = model_check(sample_imgs)
loss = nn.CrossEntropyLoss()(out, sample_labels.to(device))
loss.backward()

all_ok = all(p.grad is not None for p in model_check.parameters())
print(f"All gradients exist: {all_ok}")
print("Forward & Backward pass passed")

Output shape : torch.Size([64, 7])
Initial loss : 2.3653 (expected ~1.9459)
All gradients exist: True
Forward & Backward pass passed


In [7]:
import torch.optim as optim

def train_resnet(model, run_name, config, epochs=25):
    criterion = nn.CrossEntropyLoss()

    backbone_params = list(model.resnet.parameters())[:-2]
    head_params     = list(model.resnet.fc.parameters())

    optimizer = optim.Adam([
        {'params': backbone_params, 'lr': config.get('backbone_lr', 1e-4)},
        {'params': head_params,     'lr': config.get('head_lr', 1e-3)}
    ], weight_decay=config.get('weight_decay', 1e-4))

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    wandb.init(project="face expression recognition", name=run_name, config=config)

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += (outputs.argmax(1) == labels).sum().item()

        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                val_correct += (outputs.argmax(1) == labels).sum().item()

        train_acc  = train_correct / len(train_dataset)
        val_acc    = val_correct   / len(val_dataset)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f} | LR: {current_lr:.6f}")

        wandb.log({
            "train/loss": train_loss / len(train_loader),
            "train/accuracy": train_acc,
            "val/loss": val_loss / len(val_loader),
            "val/accuracy": val_acc,
            "learning_rate": current_lr,
            "epoch": epoch + 1
        })
        scheduler.step()

    wandb.finish()

In [8]:
model = ResNet18FER(dropout_rate=0.5).to(device)
train_resnet(model,
    run_name="ResNet18-pretrained-finetune-all",
    config={
        "architecture": "ResNet18",
        "pretrained": True,
        "backbone_lr": 1e-4,
        "head_lr": 1e-3,
        "dropout": 0.5,
        "weight_decay": 1e-4,
        "scheduler": "CosineAnnealing",
        "epochs": 15,
        "note": "pretrained resnet18, fine-tune all layers with different LRs"
    },
    epochs=15
)

Epoch 1/15 | Train Acc: 0.286 | Val Acc: 0.384 | LR: 0.000100
Epoch 2/15 | Train Acc: 0.393 | Val Acc: 0.434 | LR: 0.000099
Epoch 3/15 | Train Acc: 0.443 | Val Acc: 0.454 | LR: 0.000096
Epoch 4/15 | Train Acc: 0.479 | Val Acc: 0.484 | LR: 0.000090
Epoch 5/15 | Train Acc: 0.506 | Val Acc: 0.507 | LR: 0.000083
Epoch 6/15 | Train Acc: 0.533 | Val Acc: 0.522 | LR: 0.000075
Epoch 7/15 | Train Acc: 0.556 | Val Acc: 0.530 | LR: 0.000065
Epoch 8/15 | Train Acc: 0.582 | Val Acc: 0.541 | LR: 0.000055
Epoch 9/15 | Train Acc: 0.613 | Val Acc: 0.537 | LR: 0.000045
Epoch 10/15 | Train Acc: 0.638 | Val Acc: 0.550 | LR: 0.000035
Epoch 11/15 | Train Acc: 0.665 | Val Acc: 0.566 | LR: 0.000025
Epoch 12/15 | Train Acc: 0.688 | Val Acc: 0.557 | LR: 0.000017
Epoch 13/15 | Train Acc: 0.710 | Val Acc: 0.560 | LR: 0.000010
Epoch 14/15 | Train Acc: 0.723 | Val Acc: 0.559 | LR: 0.000004
Epoch 15/15 | Train Acc: 0.729 | Val Acc: 0.563 | LR: 0.000001


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,███▇▇▆▆▅▄▃▃▂▂▁▁
train/accuracy,▁▃▃▄▄▅▅▆▆▇▇▇███
train/loss,█▆▆▅▅▄▄▃▃▃▂▂▁▁▁
val/accuracy,▁▃▄▅▆▆▇▇▇▇█████
val/loss,█▅▄▃▂▁▁▂▁▁▁▂▂▃▃
epoch,15
learning_rate,0.0
train/accuracy,0.72873
train/loss,0.73346
val/accuracy,0.56322


In [9]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [10]:
model = ResNet18FER(dropout_rate=0.5).to(device)


for name, param in model.resnet.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

train_resnet(model,
    run_name="ResNet18-pretrained-frozen-backbone",
    config={
        "architecture": "ResNet18",
        "pretrained": True,
        "frozen_backbone": True,
        "head_lr": 1e-3,
        "dropout": 0.5,
        "scheduler": "CosineAnnealing",
        "epochs": 15,
        "note": "frozen backbone - only train classification head"
    },
    epochs=15
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/15 | Train Acc: 0.213 | Val Acc: 0.270 | LR: 0.000100
Epoch 2/15 | Train Acc: 0.235 | Val Acc: 0.258 | LR: 0.000099
Epoch 3/15 | Train Acc: 0.240 | Val Acc: 0.263 | LR: 0.000096
Epoch 4/15 | Train Acc: 0.238 | Val Acc: 0.261 | LR: 0.000090
Epoch 5/15 | Train Acc: 0.242 | Val Acc: 0.271 | LR: 0.000083
Epoch 6/15 | Train Acc: 0.244 | Val Acc: 0.261 | LR: 0.000075
Epoch 7/15 | Train Acc: 0.250 | Val Acc: 0.280 | LR: 0.000065
Epoch 8/15 | Train Acc: 0.251 | Val Acc: 0.263 | LR: 0.000055
Epoch 9/15 | Train Acc: 0.252 | Val Acc: 0.264 | LR: 0.000045
Epoch 10/15 | Train Acc: 0.257 | Val Acc: 0.273 | LR: 0.000035
Epoch 11/15 | Train Acc: 0.256 | Val Acc: 0.274 | LR: 0.000025
Epoch 12/15 | Train Acc: 0.258 | Val Acc: 0.279 | LR: 0.000017
Epoch 13/15 | Train Acc: 0.263 | Val Acc: 0.283 | LR: 0.000010
Epoch 14/15 | Train Acc: 0.266 | Val Acc: 0.287 | LR: 0.000004
Epoch 15/15 | Train Acc: 0.268 | Val Acc: 0.288 | LR: 0.000001


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,███▇▇▆▆▅▄▃▃▂▂▁▁
train/accuracy,▁▄▄▄▅▅▆▆▆▇▇▇▇██
train/loss,█▄▃▃▃▃▃▂▂▂▂▁▁▁▁
val/accuracy,▄▁▂▂▄▁▆▂▂▄▅▆▇██
val/loss,▇█▇▇▅▆▄▄▃▃▂▂▁▁▁
epoch,15
learning_rate,0.0
train/accuracy,0.26755
train/loss,1.78066
val/accuracy,0.28805


In [11]:
model = ResNet18FER(dropout_rate=0.5).to(device)

model.resnet = models.resnet18(pretrained=False)
model.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.resnet.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(512, 7))
model = model.to(device)

train_resnet(model,
    run_name="ResNet18-scratch-no-pretrained",
    config={
        "architecture": "ResNet18",
        "pretrained": False,
        "backbone_lr": 1e-3,
        "head_lr": 1e-3,
        "dropout": 0.5,
        "scheduler": "CosineAnnealing",
        "epochs": 15,
        "note": "no pretrained weights - shows value of transfer learning"
    },
    epochs=15
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 1/15 | Train Acc: 0.365 | Val Acc: 0.402 | LR: 0.001000
Epoch 2/15 | Train Acc: 0.462 | Val Acc: 0.468 | LR: 0.000989
Epoch 3/15 | Train Acc: 0.501 | Val Acc: 0.532 | LR: 0.000957
Epoch 4/15 | Train Acc: 0.533 | Val Acc: 0.547 | LR: 0.000905
Epoch 5/15 | Train Acc: 0.553 | Val Acc: 0.548 | LR: 0.000835
Epoch 6/15 | Train Acc: 0.572 | Val Acc: 0.547 | LR: 0.000750
Epoch 7/15 | Train Acc: 0.591 | Val Acc: 0.557 | LR: 0.000655
Epoch 8/15 | Train Acc: 0.616 | Val Acc: 0.581 | LR: 0.000552
Epoch 9/15 | Train Acc: 0.628 | Val Acc: 0.602 | LR: 0.000448
Epoch 10/15 | Train Acc: 0.656 | Val Acc: 0.594 | LR: 0.000345
Epoch 11/15 | Train Acc: 0.679 | Val Acc: 0.592 | LR: 0.000250
Epoch 12/15 | Train Acc: 0.705 | Val Acc: 0.612 | LR: 0.000165
Epoch 13/15 | Train Acc: 0.730 | Val Acc: 0.612 | LR: 0.000095
Epoch 14/15 | Train Acc: 0.756 | Val Acc: 0.613 | LR: 0.000043
Epoch 15/15 | Train Acc: 0.761 | Val Acc: 0.616 | LR: 0.000011


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,███▇▇▆▆▅▄▃▃▂▂▁▁
train/accuracy,▁▃▃▄▄▅▅▅▆▆▇▇▇██
train/loss,█▆▆▅▅▄▄▄▃▃▃▂▂▁▁
val/accuracy,▁▃▅▆▆▆▆▇█▇▇████
val/loss,█▄▃▂▂▂▂▁▁▁▁▁▂▂▂
epoch,15
learning_rate,1e-05
train/accuracy,0.76109
train/loss,0.65129
val/accuracy,0.61616
